In [ ]:

import sys
from pathlib import Path
import cv2
import json

import time



In [24]:
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")


from src.pipeline.table_extraction import run_table_recognition

Project root: /home/nischal/fellowship/DocuAgent-AP


/home/nischal/fellowship/DocuAgent-AP/.venv/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
which: no ccache in (/home/nischal/fellowship/DocuAgent-AP/.venv/bin:/usr/local/cuda/bin:/usr/local/cuda/bin:/usr/local/cuda/bin:/home/nischal/.local/bin:/home/nischal/bin:/usr/local/bin:/usr/bin:/bin:/usr/local/sbin:/usr/sbin:/sbin)
/home/nischal/fellowship/DocuAgent-AP/.venv/lib64/python3.13/site-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


In [25]:
import json
from pathlib import Path

json_path = Path("../data/final_eval.json")

with open(json_path, "r", encoding="utf-8") as f:
    annotations = json.load(f)

print(f"Total samples: {len(annotations)}")


Total samples: 9064


In [ ]:
img_key = list(annotations.keys())[0]  
print("Image key:", img_key)

sample_ann = annotations[img_key]
print("Keys in this annotation:", sample_ann.keys())

# Pretty print a sample
from pprint import pprint
pprint(sample_ann)


Image key: 8972e89bad9919ad83c32b48e7b9e9cd51e4789b2ccabb477e312941d5761c82.png
Keys in this annotation: dict_keys(['html', 'type'])
{'html': '<html><body><table><thead><tr><td><b>Parameters</b></td><td><b>School '
         'A</b></td><td><b>School '
         'B</b></td></tr></thead><tbody><tr><td><b><i>q</i><sub><i>c</i></sub>, '
         'mean [ 95 <i>%</i> CI]</b></td><td><b>1.39×10<sup>−2</sup> [ '
         '8.10×10<sup>−3</sup>−2.03×10<sup>−2</sup>]</b></td><td><b>1.96×10<sup>−2</sup> '
         '[ '
         '1.11×10<sup>−2</sup>−2.89×10<sup>−2</sup>]</b></td></tr><tr><td><b><i>q</i><sub><i>g</i></sub>, '
         'mean [ 95 <i>%</i> CI]</b></td><td><b>4.36×10<sup>−3</sup> [ '
         '9.61×10<sup>−4</sup>−8.34×10<sup>−3</sup>]</b></td><td><b>4.61×10<sup>−3</sup> '
         '[ '
         '2.98×10<sup>−4</sup>−1.15×10<sup>−2</sup>]</b></td></tr><tr><td><b><i>q</i><sub><i>s</i></sub>, '
         'mean [ 95 <i>%</i> CI]</b></td><td><b>9.52×10<sup>−4</sup> [ '
         '2.87×10<sup>

In [51]:
from notebooks.teds import TEDS

In [ ]:
import json
from pathlib import Path
import cv2
from src.pipeline.table_extraction import run_table_recognition

# # --- Paths ---
images_dir = Path("../data/final_eval")   # folder with images
json_path = Path("../data/final_eval.json")  # ground truth JSON


with open(json_path, "r", encoding="utf-8") as f:
    annotations = json.load(f)


all_images = [img for img in images_dir.iterdir() if img.suffix.lower() in [".png", ".jpg", ".jpeg"]]


#teds object
teds = TEDS(structure_only=True)

results = [] 

for idx, img_file in enumerate(all_images, start=1):
    filename = img_file.name
    print(f"Processing {idx}/{len(all_images)}: {filename}")
    

    image = cv2.imread(str(img_file))
    if image is None:
        print(f" Failed to load {filename}")
        continue

 
    item  = annotations.get(filename)
    if item is None or "html" not in item:
        print(f" No GT HTML for {filename}")
        continue
    gt_html = item["html"]

    #  table recognition
 
 
    pred_html = ""

    table_res= run_table_recognition(image, filename)

  
    extracted_content = [res.html for res in table_res]
    if extracted_content and "table_1" in extracted_content[0]:
            pred_html=  str(extracted_content[0]["table_1"])

    # teds computation 
    score = teds(gt_html, pred_html)
    results.append({"image": filename, "teds": score, "gt_html":gt_html,"pred_html":pred_html})
    print(f" TEDS for {filename}: {score:.4f}")

if results:
    avg_teds = sum([r["teds"] for r in results]) / len(results)
    print(f"\nAverage TEDS for {len(results)} images: {avg_teds:.5f}")


with open("teds_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)


Processing 1/280: 6c338e811af9402b98f215c9759761bc6c3def789a9ad83789f208d267d0006a.png
[INFO] Performing Table extraction on cropped regions for 6c338e811af9402b98f215c9759761bc6c3def789a9ad83789f208d267d0006a.png
[INFO] Table extraction on 6c338e811af9402b98f215c9759761bc6c3def789a9ad83789f208d267d0006a.png took 1.77 seconds
 TEDS for 6c338e811af9402b98f215c9759761bc6c3def789a9ad83789f208d267d0006a.png: 0.9490
Processing 2/280: 8be7af498b313fcdc872c170f32425cebc75098e7c2119ea1b47a88c691245eb.png
[INFO] Performing Table extraction on cropped regions for 8be7af498b313fcdc872c170f32425cebc75098e7c2119ea1b47a88c691245eb.png
[INFO] Table extraction on 8be7af498b313fcdc872c170f32425cebc75098e7c2119ea1b47a88c691245eb.png took 0.45 seconds
 TEDS for 8be7af498b313fcdc872c170f32425cebc75098e7c2119ea1b47a88c691245eb.png: 0.8750
Processing 3/280: da8512c3da0b345d6c822eda63edeec3e5bd7bbf92e06c067178beb54447138f.png
[INFO] Performing Table extraction on cropped regions for da8512c3da0b345d6c822eda6